%md
# 03_build_gold_tables — Capa Gold para señales TradingView

## 1. Objetivo de la capa Gold

La capa **Gold** convierte la tabla Silver `trading.silver.alerts_clean` en tablas orientadas a consumo final.

Mientras que Silver mantiene una vista limpia, detallada y trazable de cada evento, Gold crea tablas más específicas para:

- análisis operativo,
- dashboards,
- métricas agregadas,
- backtesting,
- Machine Learning,
- labeling futuro de outcomes.

Fuente principal:

    trading.silver.alerts_clean

Tablas destino propuestas:

    trading.gold.validated_trade_signals
    trading.gold.signal_quality_metrics
    trading.gold.ml_training_candidates

---

## 2. Rol dentro del Lakehouse

Flujo actual:

    TradingView
      ↓
    Vercel /api/webhook
      ↓
    AWS SQS
      ↓
    AWS Lambda Consumer
      ↓
    Supabase operacional
      ↓
    AWS S3 Bronze JSONL
      ↓
    Databricks Auto Loader
      ↓
    trading.bronze.alerts_raw
      ↓
    trading.silver.alerts_clean
      ↓
    03_build_gold_tables
      ↓
    trading.gold.validated_trade_signals
    trading.gold.signal_quality_metrics
    trading.gold.ml_training_candidates

La capa Gold es el punto donde los datos ya no están diseñados principalmente para trazabilidad técnica, sino para responder preguntas concretas de negocio, estrategia y entrenamiento ML.

---

## 3. Diferencia entre Silver y Gold

Silver:

- conserva granularidad completa del evento;
- mantiene `raw_payload` y `raw_validation`;
- permite auditoría;
- permite reprocesamiento;
- contiene muchas columnas técnicas;
- sirve como fuente reutilizable.

Gold:

- crea tablas enfocadas a casos de uso concretos;
- reduce complejidad para consultas;
- separa consumo operativo, métricas y ML;
- facilita dashboards;
- facilita backtesting;
- facilita datasets de entrenamiento.

---

## 4. Tabla Gold 1 — validated_trade_signals

Tabla destino:

    trading.gold.validated_trade_signals

Objetivo:

Contener una fila por señal validada relevante, con los campos necesarios para análisis operativo, dashboards y revisión de decisiones.

Incluye:

- identificación del evento,
- símbolo,
- timeframe,
- tipo de evento,
- dirección,
- precio de entrada,
- TP,
- SL,
- RR,
- aprobación,
- confianza,
- probabilidad TP antes de SL,
- score externo,
- contexto técnico,
- microestructura,
- estructura de mercado,
- timestamps de procesamiento.

Campos principales:

    signal_id
    event_uid
    symbol
    tf
    event
    side
    entry_price
    tp
    sl
    rr
    validation_approve
    validation_confidence
    probability_tp_before_sl
    score_external
    quality_score_alert
    regime
    phase
    dir_state
    mov_state
    liq_state
    htf_phase
    htf_phase_strength
    close_1m
    adx_1m
    atr14_1m
    spread_bps
    book_imbalance
    buy_aggression
    sell_aggression
    delta_qty
    last_swing_high
    last_swing_low
    distance_to_swing_high_pct
    distance_to_swing_low_pct
    processing_date
    backend_ingestion_ts
    _silver_processed_ts
    _gold_processed_ts

Uso esperado:

- dashboard de señales aprobadas/rechazadas;
- revisión de señales recientes;
- análisis de calidad de entrada;
- análisis operativo por símbolo/timeframe;
- input para backtesting.

---

## 5. Tabla Gold 2 — signal_quality_metrics

Tabla destino:

    trading.gold.signal_quality_metrics

Objetivo:

Crear métricas agregadas por fecha, símbolo, timeframe, evento, lado, fase y régimen.

Incluye:

- total de señales,
- señales aprobadas,
- señales rechazadas,
- tasa de aprobación,
- confianza media,
- probabilidad media TP antes de SL,
- score externo medio,
- quality score medio,
- ADX medio,
- spread medio,
- imbalance medio,
- agresión compradora/vendedora media.

Campos principales:

    metrics_id
    processing_date
    symbol
    tf
    event
    side
    regime
    phase
    htf_phase
    total_signals
    approved_signals
    rejected_signals
    approval_rate
    avg_validation_confidence
    avg_probability_tp_before_sl
    avg_score_external
    avg_quality_score_alert
    avg_adx_1m
    avg_spread_bps
    avg_book_imbalance
    avg_buy_aggression
    avg_sell_aggression
    avg_delta_qty
    _gold_processed_ts

Uso esperado:

- métricas para dashboard;
- comparación por setup;
- análisis por régimen y fase;
- seguimiento de calidad del sistema;
- reporting de rendimiento de señales.

---

## 6. Tabla Gold 3 — ml_training_candidates

Tabla destino:

    trading.gold.ml_training_candidates

Objetivo:

Crear una tabla específica para futuros datasets ML y proceso de labeling.

Esta tabla todavía no contiene el resultado real del trade, porque falta implementar el outcome labeling. Pero deja preparado cada candidato con:

- señal de entrada,
- ventana de labeling,
- TP,
- SL,
- reglas de ambiguous y timeout,
- features principales,
- target conceptual esperado.

Campos principales:

    candidate_id
    event_uid
    symbol
    tf
    event
    side
    candidate_type
    track_for_outcome
    labeling_profile
    ml_target
    ambiguous_rule
    timeout_rule
    entry_reference
    tp_sl_source
    entry_price
    tp
    sl
    rr
    validation_approve
    validation_confidence
    probability_tp_before_sl
    score_external
    quality_score_alert
    close_1m
    ema20_1m
    ema50_1m
    ema200_1m
    adx_1m
    plus_di_1m
    minus_di_1m
    atr14_1m
    rvol20_1m
    impulse_atr_1m
    dist_to_vwap_pct_1m
    spread_bps
    book_imbalance
    buy_aggression
    sell_aggression
    delta_qty
    regime
    phase
    dir_state
    mov_state
    liq_state
    htf_phase
    htf_phase_strength
    processing_date
    outcome_status
    outcome_label
    outcome_tp_hit_ts
    outcome_sl_hit_ts
    outcome_bars_to_resolution
    _gold_processed_ts

Uso esperado:

- construir datasets ML;
- etiquetar si TP ocurrió antes que SL;
- excluir ambiguous same bar;
- excluir o separar timeout;
- entrenar modelos como XGBoost posteriormente;
- comparar predicción vs resultado real.

---

## 7. Deduplicación en Gold

Cada tabla Gold usa una clave estable:

Para `validated_trade_signals`:

    signal_id = sha2(event_uid || message_type || event)

Para `ml_training_candidates`:

    candidate_id = sha2(event_uid || labeling_profile || ml_target)

Para `signal_quality_metrics`:

    metrics_id = sha2(processing_date || symbol || tf || event || side || regime || phase || htf_phase)

Esto permite ejecuciones repetidas sin duplicar datos.

---

## 8. Estrategia de escritura

La capa Gold usa dos patrones:

1. `MERGE` incremental para tablas de detalle:

       trading.gold.validated_trade_signals
       trading.gold.ml_training_candidates

2. `overwrite` controlado para tabla agregada:

       trading.gold.signal_quality_metrics

La tabla de métricas puede recalcularse completa desde Silver porque es una agregación derivada. Esto simplifica la consistencia.

---

## 9. Estado actual esperado

VALIDADO:

- Bronze ya ingiere desde S3.
- Silver ya transforma y aplana correctamente.
- Supabase y S3 funcionan.
- Lambda procesa CORE + EXTRA.
- `trading.silver.alerts_clean` está validada.

PROPUESTA:

- Crear capa Gold con tres tablas iniciales:
  - `validated_trade_signals`
  - `signal_quality_metrics`
  - `ml_training_candidates`

PENDIENTE:

- Implementar outcome labeling.
- Crear Gold de performance real.
- Crear Gold de backtesting.
- Crear dataset final de entrenamiento ML.
- Crear job programado para Bronze → Silver → Gold.

---

## 10. Resumen ejecutivo

La capa `03_build_gold_tables` toma la tabla Silver ya validada y construye tres salidas principales:

    trading.gold.validated_trade_signals
    trading.gold.signal_quality_metrics
    trading.gold.ml_training_candidates

Estas tablas dejan el Lakehouse preparado para:

- dashboards de señales,
- análisis de calidad,
- métricas por setup,
- backtesting,
- labeling,
- entrenamiento ML,
- futuras tablas de performance.

---


In [0]:
# ============================================================
# 03_build_gold_tables
# Construcción de tablas Gold desde Silver
#
# Fuente:
#   trading.silver.alerts_clean
#
# Destinos:
#   trading.gold.validated_trade_signals
#   trading.gold.signal_quality_metrics
#   trading.gold.ml_training_candidates
#
# Objetivo:
# - Crear tablas orientadas a consumo final.
# - Preparar datos para dashboards, análisis, ML y backtesting.
# - Mantener idempotencia mediante claves técnicas.
# ============================================================

from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ============================================================
# CONFIG
# ============================================================

SOURCE_TABLE = "trading.silver.alerts_clean"

GOLD_SCHEMA = "trading.gold"

TARGET_VALIDATED_SIGNALS = "trading.gold.validated_trade_signals"
TARGET_SIGNAL_METRICS = "trading.gold.signal_quality_metrics"
TARGET_ML_CANDIDATES = "trading.gold.ml_training_candidates"

print("SOURCE_TABLE:", SOURCE_TABLE)
print("TARGET_VALIDATED_SIGNALS:", TARGET_VALIDATED_SIGNALS)
print("TARGET_SIGNAL_METRICS:", TARGET_SIGNAL_METRICS)
print("TARGET_ML_CANDIDATES:", TARGET_ML_CANDIDATES)

# ============================================================
# ENSURE GOLD SCHEMA
# ============================================================

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

# ============================================================
# READ SILVER
# ============================================================

df_silver = spark.table(SOURCE_TABLE)

print("Silver rows:", df_silver.count())

# ============================================================
# HELPER: MERGE INTO DELTA TABLE
# ============================================================

def merge_to_delta(source_df, target_table: str, merge_key: str) -> None:
    clean_df = (
        source_df
        .filter(F.col(merge_key).isNotNull())
        .dropDuplicates([merge_key])
    )

    if clean_df.isEmpty():
        print(f"No rows to merge into {target_table}")
        return

    if not spark.catalog.tableExists(target_table):
        (
            clean_df
            .write
            .format("delta")
            .mode("overwrite")
            .option("mergeSchema", "true")
            .saveAsTable(target_table)
        )

        print(f"Created table {target_table}")
        return

    target_columns = spark.table(target_table).columns
    source_columns = clean_df.columns

    missing_columns = [c for c in source_columns if c not in target_columns]

    if missing_columns:
        print(f"Detected new columns in source for {target_table}: {missing_columns}")
        print(f"Applying schema evolution using append + mergeSchema before MERGE.")

        (
            clean_df
            .limit(0)
            .write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(target_table)
        )

    target = DeltaTable.forName(spark, target_table)

    (
        target.alias("t")
        .merge(
            clean_df.alias("s"),
            f"t.{merge_key} = s.{merge_key}"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Merged rows into {target_table}")
# ============================================================
# GOLD 1: VALIDATED TRADE SIGNALS
# ============================================================

df_validated_signals = (
    df_silver
    .filter(F.col("event_uid").isNotNull())
    .filter(F.col("message_type") == "logical_event_full")
    .withColumn(
        "signal_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("event_uid"), F.lit("")),
                F.coalesce(F.col("message_type"), F.lit("")),
                F.coalesce(F.col("event"), F.lit(""))
            ),
            256
        )
    )
    .select(
        "signal_id",
        "event_uid",
        "schema_version",
        "message_type",
        "trace_id",
        "symbol",
        "symbol_normalized",
        "tf",
        "event",
        "side",
        "side_normalized",
        "signal_time",
        "entry_price",
        "tp",
        "sl",
        "rr",
        "validation_approve",
        "validation_confidence",
        "probability_tp_before_sl",
        "score_external",
        "quality_score_alert",
        "probability_model",
        "barrier_component",
        "technical_component",
        "ml_component",
        "regime",
        "phase",
        "dir_state",
        "mov_state",
        "liq_state",
        "htf_phase",
        "htf_phase_strength",
        "trigger_alignment",
        "too_extended_warn_alert",
        "too_extended_block_alert",
        "late_trend_alert",
        "close_1m",
        "ema20_1m",
        "ema50_1m",
        "ema200_1m",
        "adx_1m",
        "plus_di_1m",
        "minus_di_1m",
        "atr14_1m",
        "rvol20_1m",
        "impulse_atr_1m",
        "dist_to_vwap_pct_1m",
        "spread_bps",
        "book_imbalance",
        "buy_aggression",
        "sell_aggression",
        "delta_qty",
        "bid_wall_detected",
        "ask_wall_detected",
        "vacuum_above",
        "vacuum_below",
        "last_swing_high",
        "last_swing_low",
        "distance_to_swing_high_pct",
        "distance_to_swing_low_pct",
        "range_mode",
        "compression_box",
        "track_for_outcome",
        "candidate_type",
        "labeling_profile",
        "ml_target",
        "backend_ingestion_ts",
        "processing_date",
        "_databricks_ingestion_ts",
        "_source_file",
        "_source_file_modification_time",
        "_silver_processed_ts",
        F.current_timestamp().alias("_gold_processed_ts")
    )
)

merge_to_delta(
    source_df=df_validated_signals,
    target_table=TARGET_VALIDATED_SIGNALS,
    merge_key="signal_id"
)

# ============================================================
# GOLD 2: SIGNAL QUALITY METRICS
# ============================================================

df_signal_metrics = (
    df_silver
    .filter(F.col("event_uid").isNotNull())
    .filter(F.col("message_type") == "logical_event_full")
    .groupBy(
        "processing_date",
        "symbol",
        "symbol_normalized",
        "tf",
        "event",
        "side",
        "side_normalized",
        "regime",
        "phase",
        "htf_phase"
    )
    .agg(
        F.count("*").alias("total_signals"),
        F.sum(F.when(F.col("validation_approve") == True, 1).otherwise(0)).alias("approved_signals"),
        F.sum(F.when(F.col("validation_approve") == False, 1).otherwise(0)).alias("rejected_signals"),
        F.avg(F.col("validation_confidence")).alias("avg_validation_confidence"),
        F.avg(F.col("probability_tp_before_sl")).alias("avg_probability_tp_before_sl"),
        F.avg(F.col("score_external")).alias("avg_score_external"),
        F.avg(F.col("quality_score_alert")).alias("avg_quality_score_alert"),
        F.avg(F.col("adx_1m")).alias("avg_adx_1m"),
        F.avg(F.col("atr14_1m")).alias("avg_atr14_1m"),
        F.avg(F.col("rvol20_1m")).alias("avg_rvol20_1m"),
        F.avg(F.col("spread_bps")).alias("avg_spread_bps"),
        F.avg(F.col("book_imbalance")).alias("avg_book_imbalance"),
        F.avg(F.col("buy_aggression")).alias("avg_buy_aggression"),
        F.avg(F.col("sell_aggression")).alias("avg_sell_aggression"),
        F.avg(F.col("delta_qty")).alias("avg_delta_qty"),
        F.max(F.col("_silver_processed_ts")).alias("last_silver_processed_ts")
    )
    .withColumn(
        "approval_rate",
        F.when(
            F.col("total_signals") > 0,
            F.col("approved_signals") / F.col("total_signals")
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "metrics_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("processing_date").cast("string"), F.lit("")),
                F.coalesce(F.col("symbol"), F.lit("")),
                F.coalesce(F.col("tf"), F.lit("")),
                F.coalesce(F.col("event"), F.lit("")),
                F.coalesce(F.col("side"), F.lit("")),
                F.coalesce(F.col("regime"), F.lit("")),
                F.coalesce(F.col("phase"), F.lit("")),
                F.coalesce(F.col("htf_phase"), F.lit(""))
            ),
            256
        )
    )
    .withColumn("_gold_processed_ts", F.current_timestamp())
    .select(
        "metrics_id",
        "processing_date",
        "symbol",
        "symbol_normalized",
        "tf",
        "event",
        "side",
        "side_normalized",
        "regime",
        "phase",
        "htf_phase",
        "total_signals",
        "approved_signals",
        "rejected_signals",
        "approval_rate",
        "avg_validation_confidence",
        "avg_probability_tp_before_sl",
        "avg_score_external",
        "avg_quality_score_alert",
        "avg_adx_1m",
        "avg_atr14_1m",
        "avg_rvol20_1m",
        "avg_spread_bps",
        "avg_book_imbalance",
        "avg_buy_aggression",
        "avg_sell_aggression",
        "avg_delta_qty",
        "last_silver_processed_ts",
        "_gold_processed_ts"
    )
)

(
    df_signal_metrics
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_SIGNAL_METRICS)
)

print(f"Overwritten table {TARGET_SIGNAL_METRICS}")

# ============================================================
# GOLD 3: ML TRAINING CANDIDATES
# ============================================================

df_ml_candidates = (
    df_silver
    .filter(F.col("event_uid").isNotNull())
    .filter(F.col("message_type") == "logical_event_full")
    .filter(F.coalesce(F.col("track_for_outcome"), F.lit(False)) == True)
    .withColumn(
        "candidate_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("event_uid"), F.lit("")),
                F.coalesce(F.col("labeling_profile"), F.lit("")),
                F.coalesce(F.col("ml_target"), F.lit(""))
            ),
            256
        )
    )
    .withColumn("outcome_status", F.lit("PENDING_LABELING"))
    .withColumn("outcome_label", F.lit(None).cast("string"))
    .withColumn("outcome_tp_hit_ts", F.lit(None).cast("timestamp"))
    .withColumn("outcome_sl_hit_ts", F.lit(None).cast("timestamp"))
    .withColumn("outcome_bars_to_resolution", F.lit(None).cast("int"))
    .select(
        "candidate_id",
        "event_uid",
        "schema_version",
        "message_type",
        "trace_id",
        "symbol",
        "symbol_normalized",
        "tf",
        "event",
        "side",
        "side_normalized",
        "signal_time",
        "track_for_outcome",
        "candidate_type",
        "labeling_profile",
        "ml_target",
        "ambiguous_rule",
        "timeout_rule",
        "entry_reference",
        "tp_sl_source",
        "entry_price",
        "tp",
        "sl",
        "rr",
        "validation_approve",
        "validation_confidence",
        "probability_tp_before_sl",
        "score_external",
        "quality_score_alert",
        "probability_model",
        "barrier_component",
        "technical_component",
        "ml_component",
        "close_1m",
        "ema20_1m",
        "ema50_1m",
        "ema200_1m",
        "adx_1m",
        "plus_di_1m",
        "minus_di_1m",
        "atr14_1m",
        "rvol20_1m",
        "impulse_atr_1m",
        "dist_to_vwap_pct_1m",
        "spread_bps",
        "book_imbalance",
        "buy_aggression",
        "sell_aggression",
        "delta_qty",
        "regime",
        "phase",
        "dir_state",
        "mov_state",
        "liq_state",
        "htf_phase",
        "htf_phase_strength",
        "last_swing_high",
        "last_swing_low",
        "distance_to_swing_high_pct",
        "distance_to_swing_low_pct",
        "processing_date",
        "backend_ingestion_ts",
        "_databricks_ingestion_ts",
        "_silver_processed_ts",
        "outcome_status",
        "outcome_label",
        "outcome_tp_hit_ts",
        "outcome_sl_hit_ts",
        "outcome_bars_to_resolution",
        F.current_timestamp().alias("_gold_processed_ts")
    )
)

merge_to_delta(
    source_df=df_ml_candidates,
    target_table=TARGET_ML_CANDIDATES,
    merge_key="candidate_id"
)

# ============================================================
# VALIDATION QUERIES
# ============================================================

print("Gold build finished.")

print("validated_trade_signals rows:")
spark.sql(f"SELECT COUNT(*) AS total FROM {TARGET_VALIDATED_SIGNALS}").show()

print("signal_quality_metrics rows:")
spark.sql(f"SELECT COUNT(*) AS total FROM {TARGET_SIGNAL_METRICS}").show()

print("ml_training_candidates rows:")
spark.sql(f"SELECT COUNT(*) AS total FROM {TARGET_ML_CANDIDATES}").show()

## Consultas de validación

In [0]:
%sql
-- 1. Últimas señales Gold

    SELECT
      event_uid,
      symbol,
      tf,
      event,
      side,
      entry_price,
      tp,
      sl,
      rr,
      validation_approve,
      validation_confidence,
      probability_tp_before_sl,
      score_external,
      regime,
      phase,
      htf_phase,
      _gold_processed_ts
    FROM trading.gold.validated_trade_signals
    ORDER BY _gold_processed_ts DESC
    LIMIT 20;

-- 2. Métricas agregadas

    SELECT
      processing_date,
      symbol,
      tf,
      event,
      side,
      regime,
      phase,
      htf_phase,
      total_signals,
      approved_signals,
      rejected_signals,
      approval_rate,
      avg_validation_confidence,
      avg_probability_tp_before_sl,
      avg_score_external
    FROM trading.gold.signal_quality_metrics
    ORDER BY processing_date DESC, total_signals DESC;

-- 3. Candidatos ML pendientes de labeling

    SELECT
      candidate_id,
      event_uid,
      symbol,
      tf,
      event,
      side,
      candidate_type,
      labeling_profile,
      ml_target,
      entry_price,
      tp,
      sl,
      validation_approve,
      probability_tp_before_sl,
      score_external,
      outcome_status
    FROM trading.gold.ml_training_candidates
    ORDER BY _gold_processed_ts DESC
    LIMIT 20;

-- 4. Evento específico de prueba

    SELECT *
    FROM trading.gold.validated_trade_signals
    WHERE event_uid = 'test_s3_final_010';

/* Resultado esperado

Después de ejecutar el notebook deben existir las siguientes tablas:

    trading.gold.validated_trade_signals
    trading.gold.signal_quality_metrics
    trading.gold.ml_training_candidates

Y deben aparecer registros provenientes de:

    trading.silver.alerts_clean

Si el evento `test_s3_final_010` está en Silver, debería aparecer al menos en:

    trading.gold.validated_trade_signals

Y si `track_for_outcome = true`, también en:

    trading.gold.ml_training_candidates*/